# Benchmark: polynomial filters in U -- U^n vs T_n(U)

Compares two ways of turning the one-drive-period propagator `U` into a
polynomial filter that isolates the slowest decaying eigenvalue, as a
function of the polynomial degree `n = 1..10`, at a single `alpha_sq = 6`
(other physical/numerical parameters taken from `notebooks/benchmarks/benchmark_methods.ipynb`):

| key | polynomial | how it's realized |
|---|---|---|
| `power` | `U^n` | plain (unfiltered) Arnoldi on an `n`-period block, i.e. `ArnoldiLindblad` built with `n_periods = n` -- one Arnoldi step directly applies `U^n` |
| `cheb` | `T_n(U)` | `ChebAr` built on a single-period block (`n_periods = 1`) with `cheb_degree = n` -- one Arnoldi step applies the degree-`n` Chebyshev filter of `U` |

Both run the same fixed number of Arnoldi steps `m` for every `n`, so `n` is
the only thing that changes within each method as it sweeps 1..10. `ChebAr`
additionally pays a one-off `m_cheb_0`-step unfiltered estimation (needed to
fit the enclosing ellipse) before the filter can be built at all -- that cost
does not depend on `n` and is charged once per `n`.

## Cost axis

Both curves are compared in units of **base (single-period) propagator
applications**, so the `n`-dependence of the cost is explicit and comparable
across methods:

```
apps_power(n) = n * m                 # one U^n application = n base applications
apps_cheb(n)  = m_cheb_0 + n * m      # one T_n(U) application = n base applications
```

(`m_cheb_0` itself runs on the 1-period block, i.e. already in base-application
units.) Wall clock is also recorded, from a timed call to each block's own
propagator (`propagate` for `power`, `propagate_block_projected` for `cheb`).

## Reference

The most-converged answer available -- `power` at `n = 10` (the largest,
therefore most power-method-filtered, block) -- is used as the reference rate
to measure both curves against. This mirrors `notebooks/benchmarks/benchmark_methods.ipynb`, which
uses the unfiltered `arnoldi` (there, a fixed 6-period block) as its reference.

## Running it

Same requirements as `notebooks/benchmarks/benchmark_methods.ipynb`: the workspace pixi kernel
(Python 3.13.x, `ray 2.56.0`), one GPU job, run all cells in order. Expect
this to take longer than `notebooks/benchmarks/benchmark_methods.ipynb`: it evaluates 10 values of
`n` for each of two methods, each running `m` Arnoldi steps of a block that
grows with `n` for `power` (an `n`-period block costs roughly `n` times a
single application).


In [ ]:
import os
import subprocess
import sys

assert sys.version_info[:2] == (3, 13), (
    f"kernel runs Python {sys.version.split()[0]}; the cluster needs 3.13.x. "
    "Select the workspace pixi environment as the kernel."
)

# Ray reads RAY_AUTH_MODE once, at its first import, and caches it. Setting it
# here works *only* because nothing has imported ray yet in a fresh kernel. If
# you have already run anything that imports ray, restart the kernel first.
os.environ["RAY_AUTH_MODE"] = "token"
os.environ.setdefault("VAULT_ADDR", "https://vault.int.alice-bob.com")

# Mint the cluster token from Vault here, so the notebook also works from a
# kernel that was not launched from a shell with RAY_AUTH_TOKEN exported (which
# is the normal case in VS Code). `vault login` is interactive, so it stays a
# manual, once-a-week step.
if not os.environ.get("RAY_AUTH_TOKEN"):
    try:
        os.environ["RAY_AUTH_TOKEN"] = subprocess.run(
            ["vault", "kv", "get", "-field=token", "kv-it/prod/ray/auth-token"],
            capture_output=True,
            text=True,
            check=True,
        ).stdout.strip()
    except (FileNotFoundError, subprocess.CalledProcessError) as exc:
        raise RuntimeError(
            "Could not mint the cluster token from Vault. In a terminal, run:\n"
            "  export VAULT_ADDR=https://vault.int.alice-bob.com\n"
            "  vault login -method=oidc role=research_core_user\n"
            "then re-run this cell (no kernel restart needed).\n"
            f"{(getattr(exc, 'stderr', '') or '').strip()}"
        ) from exc

import ray

from anb_compute import ray as acr

print("python", sys.version.split()[0], "| ray", ray.__version__)

import pathlib


def _find_repo():
    """Locate the repo checkout without hardcoding an absolute path.

    Notebooks have no __file__, so search upward from the working directory --
    which VS Code sets to either the notebook's folder or the workspace root,
    depending on `jupyter.notebookFileRoot`. Both are handled.
    """
    here = pathlib.Path.cwd().resolve()
    for base in (here, *here.parents):
        if (base / "src" / "floquet_lindblad").is_dir():
            return base
        for hit in sorted(base.glob("*/src/floquet_lindblad")):
            return hit.parents[1]
    raise RuntimeError(f"no checkout with src/floquet_lindblad found above {here}")


REPO = _find_repo()

# Shipped by PATH, not by name: a bare module name that resolves to a
# non-editable site-packages install ships a frozen copy, and local edits would
# never reach the worker. These prepend to the worker's sys.path, so the
# dynamiqs here shadows the image's stock 0.3.6 (which lacks `mesolve_fast`).
PKG = str(REPO / "src" / "floquet_lindblad")
# The patched dynamiqs checkout, expected beside the repo; $DYNAMIQS_SRC wins.
DYNAMIQS = os.environ.get("DYNAMIQS_SRC") or str(
    REPO.parent / "dynamiqs" / "dynamiqs"
)
for _p in (PKG, DYNAMIQS):
    if not pathlib.Path(_p).is_dir():
        raise RuntimeError(f"not a directory, cannot ship to workers: {_p}")

RUNTIME_ENV = acr.build_runtime_env(py_modules=[PKG, DYNAMIQS])

In [ ]:
def run_benchmark_poly(
    alpha_sq=6.0,
    n_a=25,
    n_b=11,
    epsilon_p=0.1,
    n_values=tuple(range(1, 11)),
    m=60,
    # ChebAr
    m_cheb_0=60,
    margin=1e-2,
    seed=0,
):
    """Run both polynomial filters, U^n and T_n(U), for every n in `n_values`.

    Everything happens in one job/process/GPU so the timings across n and
    across methods are directly comparable (same device, same warm caches).

    Returns plain numpy/python objects only -- the client has neither jax nor
    dynamiqs; the Krylov basis never crosses the wire.
    """
    import time
    import traceback

    import numpy as np

    import dynamiqs as dq
    from floquet_lindblad import ArnoldiLindblad, ChebAr
    from floquet_lindblad.models.ats import build_ats_hamiltonian_interaction

    def py(v):
        """0-d jax/numpy scalar -> plain python float/complex."""
        a = np.asarray(v)
        return complex(a) if np.iscomplexobj(a) else float(a)

    def build(np_periods):
        return build_ats_hamiltonian_interaction(
            n_a=n_a,
            n_b=n_b,
            alpha_sq=alpha_sq,
            epsilon_p=epsilon_p,
            n_periods=np_periods,
        )

    def time_apply(fn, x, repeats=3):
        fn(x).block_until_ready()  # warm-up: trace + compile
        t = time.time()
        for _ in range(repeats):
            y = fn(x)
        y.block_until_ready()
        return (time.time() - t) / repeats

    out = {"power": {}, "cheb": {}}
    params = None

    for n in n_values:
        # -------------------------------------------------------------- #
        # U^n: plain Arnoldi on an n-period block                         #
        # -------------------------------------------------------------- #
        try:
            t0 = time.time()
            H_I, jops, jops_LdL, phase, V, T_block, params = build(n)
            solver = ArnoldiLindblad(
                H_I, jops, T_block, jump_ops_LdL=jops_LdL,
                dims=(n_a, n_b), output_phase=phase,
            )
            x0 = solver.make_x0(seed=seed)
            t_build = time.time() - t0

            t0 = time.time()
            Q, H, steps, mu_list = solver.arnoldi(x0, m, stride=1)
            t_run = time.time() - t0

            m_reached = int(steps[-1])
            x_ritz, _ = solver.ritz_vector(Q, H, m_reached, target=mu_list[-1])
            res = solver.residual_check(x_ritz)
            H = np.asarray(H)
            del Q, x_ritz

            out["power"][n] = {
                "n_periods": n,
                "T_block": py(T_block),
                "apps": n * m_reached,
                "mu": py(mu_list[-1]),
                "t_apply": time_apply(solver.propagate, x0),
                "t_build": t_build,
                "t_run": t_run,
                "m": m_reached,
                "mu_RQ": py(res["mu_RQ"]),
                "rate_RQ": py(res["rate_RQ"]),
                "res_rel": py(res["res_rel"]),
                "H_imag_max": float(np.abs(H.imag).max()),
            }
            del solver
        except Exception:
            out["power"][n] = {"error": traceback.format_exc()}

        # -------------------------------------------------------------- #
        # T_n(U): ChebAr on a single-period block, cheb_degree = n        #
        # -------------------------------------------------------------- #
        try:
            t0 = time.time()
            H_I, jops, jops_LdL, phase, V_cheb, T_cheb, params = build(1)
            solver = ChebAr(
                H_I, jops, T_cheb, jump_ops_LdL=jops_LdL,
                dims=(n_a, n_b), output_phase=phase, cheb_degree=n,
            )
            x0 = solver.make_x0(seed=seed)
            t_build = time.time() - t0

            t0 = time.time()
            _, _, ritz_vals = solver.first_estimation(x0, m_arnoldi=m_cheb_0)
            t_est = time.time() - t0

            solver.setup_chebyshev(ritz_vals, margin=margin)

            t0 = time.time()
            Q, H, mu_list = solver.arnoldi_hessenberg(
                x0, solver.chebyshev_filter, m, warm_start=False
            )
            t_run = time.time() - t0

            x_ritz, _ = solver.ritz_vector(Q, H, m)
            res = solver.residual_check(x_ritz)
            del Q, x_ritz

            out["cheb"][n] = {
                "n_periods": 1,
                "cheb_degree": n,
                "T_block": py(T_cheb),
                "apps": m_cheb_0 + n * m,
                "mu": py(mu_list[-1]),
                "t_apply": time_apply(solver.propagate_block_projected, x0),
                "t_apply_filter": time_apply(solver.chebyshev_filter, x0),
                "t_build": t_build,
                "t_setup": t_est,
                "t_run": t_run,
                "m": m,
                "mu_RQ": py(res["mu_RQ"]),
                "rate_RQ": py(res["rate_RQ"]),
                "res_rel": py(res["res_rel"]),
                "ellipse": {
                    k: py(v) for k, v in solver.ellipse.items() if np.ndim(v) == 0
                },
            }
            del solver
        except Exception:
            out["cheb"][n] = {"error": traceback.format_exc()}

    import jax

    return {
        "methods": out,
        "n_values": list(n_values),
        "params": None if params is None else {k: py(v) for k, v in params.items()},
        "settings": {
            "alpha_sq": alpha_sq,
            "n_a": n_a,
            "n_b": n_b,
            "epsilon_p": epsilon_p,
            "m": m,
            "m_cheb_0": m_cheb_0,
            "margin": margin,
            "seed": seed,
        },
        "device": str(jax.devices()[0]),
    }


In [ ]:
import time

# One job, one GPU: both methods and all 10 values of n share a device and a
# process, which is what makes their timings comparable at all.
t0 = time.time()
result = acr.run(
    run_benchmark_poly,
    alpha_sq=6.0,
    num_gpus=1,
    num_cpus=8,
    runtime_env=RUNTIME_ENV,
    poll_seconds=5,
)
print(f"{time.time() - t0:.1f} s wall (incl. startup)")
print("device:", result["device"])

for method in ("power", "cheb"):
    print(f"\n{method}:")
    for n, m in result["methods"][method].items():
        if "error" in m:
            print(f"  n={n:2d} FAILED")
        else:
            print(
                f"  n={n:2d}  apps={m['apps']:5d}  "
                f"t_run={m['t_run']:7.1f} s  t_apply={m['t_apply'] * 1e3:7.1f} ms"
            )

# Surface any failure in full rather than letting the plots go quietly empty.
for method in ("power", "cheb"):
    for n, m in result["methods"][method].items():
        if "error" in m:
            print(f"\n=== {method}, n={n} ===\n{m['error']}")


In [ ]:
import numpy as np

STYLE = {
    "power": {"color": "#E69F00", "marker": "s", "ls": "--", "label": r"$U^n$ (power, unfiltered)"},
    "cheb": {"color": "#0072B2", "marker": "o", "ls": "-", "label": r"$T_n(U)$ (Chebyshev filter)"},
}


def rate_of(m):
    """Predicted bit-flip rate, `-log(mu) / T_block`."""
    return (-np.log(complex(m["mu"])) / m["T_block"]).real


def ok(method):
    """{n: entry} for the n that did not raise, in n order."""
    return {
        n: m
        for n, m in sorted(result["methods"][method].items(), key=lambda kv: kv[0])
        if "error" not in m
    }


power_ok = ok("power")
cheb_ok = ok("cheb")

# Reference: the largest-n power-method result (most-converged block available).
n_ref = max(power_ok)
ref = power_ok[n_ref]["rate_RQ"]
print(f"reference: power (U^n), n={n_ref}, rate_RQ = {ref:.6e}\n")

print(f"{'method':6s} {'n':>3} {'apps':>6} {'rate_bf':>13} {'rel. dev.':>10} {'res_rel':>10} {'run [s]':>9}")
for method, entries in (("power", power_ok), ("cheb", cheb_ok)):
    for n, m in entries.items():
        rate = rate_of(m)
        print(
            f"{method:6s} {n:3d} {m['apps']:6d} {rate:13.6e} "
            f"{abs(rate - ref) / abs(ref):10.2e} {abs(m['res_rel']):10.2e} "
            f"{m['t_run']:9.1f}"
        )


In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(6.0, 4.2), constrained_layout=True)

for method, entries in (("power", power_ok), ("cheb", cheb_ok)):
    ns = sorted(entries)
    rates = [abs(rate_of(entries[n])) for n in ns]
    ax.plot(ns, rates, color=STYLE[method]["color"], marker=STYLE[method]["marker"],
            ls=STYLE[method]["ls"], label=STYLE[method]["label"], ms=5, lw=1.4)

ax.axhline(abs(ref), color="0.35", lw=0.8, ls="-", zorder=0)
ax.annotate("converged rate", xy=(0.98, abs(ref)), xycoords=("axes fraction", "data"),
            ha="right", va="bottom", fontsize=8, color="0.35")
ax.set_yscale("log")
ax.set_xlabel("polynomial degree $n$")
ax.set_ylabel(r"predicted bit-flip rate $|{-}\log\mu / T_{\rm block}|$")
ax.set_title(r"Predicted bit-flip rate vs polynomial degree ($|\alpha|^2=6$)", fontsize=10)
ax.legend(fontsize=8, frameon=False)
for side in ("top", "right"):
    ax.spines[side].set_visible(False)
ax.grid(True, which="major", lw=0.4, color="0.9")
ax.set_axisbelow(True)
plt.show()


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11.0, 4.2), constrained_layout=True)

ax = axes[0]
for method, entries in (("power", power_ok), ("cheb", cheb_ok)):
    ns = sorted(entries)
    err = [abs(rate_of(entries[n]) - ref) / abs(ref) for n in ns]
    ax.plot(ns, err, color=STYLE[method]["color"], marker=STYLE[method]["marker"],
            ls=STYLE[method]["ls"], label=STYLE[method]["label"], ms=5, lw=1.4)
ax.set_yscale("log")
ax.set_xlabel("polynomial degree $n$")
ax.set_ylabel("relative deviation from the converged rate")
ax.set_title("Convergence vs degree", fontsize=10)
ax.legend(fontsize=8, frameon=False)

ax = axes[1]
for method, entries in (("power", power_ok), ("cheb", cheb_ok)):
    ns = sorted(entries)
    apps = [entries[n]["apps"] for n in ns]
    err = [abs(rate_of(entries[n]) - ref) / abs(ref) for n in ns]
    ax.plot(apps, err, color=STYLE[method]["color"], marker=STYLE[method]["marker"],
            ls=STYLE[method]["ls"], label=STYLE[method]["label"], ms=5, lw=1.4)
ax.set_yscale("log")
ax.set_xlabel("base-period propagator applications")
ax.set_ylabel("relative deviation from the converged rate")
ax.set_title("Convergence vs cost", fontsize=10)

for ax in axes:
    for side in ("top", "right"):
        ax.spines[side].set_visible(False)
    ax.grid(True, which="major", lw=0.4, color="0.9")
    ax.set_axisbelow(True)
plt.show()


In [ ]:
fig, ax = plt.subplots(figsize=(6.0, 4.2), constrained_layout=True)

for method, entries in (("power", power_ok), ("cheb", cheb_ok)):
    ns = sorted(entries)
    t_run = [entries[n]["t_run"] for n in ns]
    ax.plot(ns, t_run, color=STYLE[method]["color"], marker=STYLE[method]["marker"],
            ls=STYLE[method]["ls"], label=STYLE[method]["label"], ms=5, lw=1.4)

ax.set_xlabel("polynomial degree $n$")
ax.set_ylabel("measured Arnoldi run time [s]")
ax.set_title(r"Measured wall clock vs polynomial degree ($|\alpha|^2=6$)", fontsize=10)
ax.legend(fontsize=8, frameon=False)
for side in ("top", "right"):
    ax.spines[side].set_visible(False)
ax.grid(True, which="major", lw=0.4, color="0.9")
ax.set_axisbelow(True)
plt.show()


## How to read this

- **Rate-vs-degree panel** -- both curves should approach the same horizontal
  line as `n` grows; `T_n(U)` is expected to approach it much faster than
  `U^n` for the same `n`, since a Chebyshev filter suppresses the unwanted
  part of the spectrum exponentially in `n` while the plain power `U^n` only
  suppresses it geometrically in the ratio of eigenvalue magnitudes.
- **Convergence-vs-cost panel** is the one that matters: it renders both
  curves in the same currency (base-period propagator applications), so a
  crossing or gap here is the actual case for (or against) filtering, at
  fixed `n`.
- **Wall-clock panel** is diagnostic, not comparative across methods at fixed
  `n`: `cheb` at degree `n` and `power` at `n_periods = n` apply the
  propagator a similar number of times, so if `t_run` diverges between them
  at the same `n`, look at `t_apply` / `t_apply_filter` for why (jit/tracing
  overhead of the Chebyshev recurrence vs. a longer single `mesolve` call).
- As in `notebooks/benchmarks/benchmark_methods.ipynb`, `res_rel` alone is not sufficient evidence
  of a correct answer for `cheb` -- always check the rate against the
  reference too.
